# Age Conclusions

Builds yes/no totals for the four age buckets (Children, Young Adult, Middle Aged, Elderly) using the **same `groups` mapping as `conclusions/get_conclusions.ipynb`**. Renders a stacked horizontal bar chart and saves it as both PNG and SVG. The companion notebook `age_distributions.ipynb` renders a frequency-only bar chart from the same data.

In [1]:
import json
import re
import sys
from pathlib import Path

import pandas as pd

_IMGS = Path.cwd() / "imgs"
_IMGS.mkdir(parents=True, exist_ok=True)
_REPO = _IMGS.parents[2]  # .../viz
sys.path.insert(0, str(_REPO / "conclusions"))

from seaborn_bar_utils import render_stacked_horizontal

In [2]:
df = pd.read_csv(_REPO / "data" / "papers.csv")

paper_conclusions_lst = df["Conclusions"].tolist()
conclusions_mps: dict[str, dict[str, int]] = {}

for paper in paper_conclusions_lst:
    if isinstance(paper, float):
        continue

    for conclusion in str(paper).split(","):
        try:
            value = re.sub(r"\s*\([^)]*\)", "", conclusion).strip()
            key, val = (part.strip() for part in value.split(":", 1))
            if not key or not val:
                continue

            key_norm = key.strip()
            yn = val.strip().lower().split(None, 1)[0]

            if key_norm not in conclusions_mps:
                conclusions_mps[key_norm] = {"yes": 0, "no": 0}

            if yn == "yes":
                conclusions_mps[key_norm]["yes"] += 1
            elif yn == "no":
                conclusions_mps[key_norm]["no"] += 1
        except ValueError:
            continue

len(conclusions_mps)

204

In [3]:
groups = {
    "Children": ['children', 'youth', 'Youth', 'Young', 'Child', 'Younger'],
    "Young Adult": ['25yo', 'Age <20', 'teenager', '18-40', 'Young Adult',
                    'Age 18 to 30'],
    "Middle Aged": ['Age 40', 'Age 80', 'Age 20-40', 'Age 40-60',
                    '40-60', '<40', '40–50', '50–60', '>60', '55 Years',
                    'middle-aged', 'Mid-age', 'Middle Adult', 'Older adults'],
    "Elderly": ['75yo', 'Age 60-80', 'Age >80', '60-80', '>80',
                '65 Years', 'elderly', 'Older Middle-Aged',
                'Middle-Aged', 'Old', 'Age 51+', 'Elderly'],
}

age_conclusions: dict[str, dict[str, int]] = {}
for new_name, categories in groups.items():
    yes_total = sum(conclusions_mps.get(cat, {"yes": 0})["yes"] for cat in categories)
    no_total = sum(conclusions_mps.get(cat, {"no": 0})["no"] for cat in categories)
    age_conclusions[new_name] = {"yes": yes_total, "no": no_total}

age_conclusions

{'Children': {'yes': 5, 'no': 4},
 'Young Adult': {'yes': 3, 'no': 3},
 'Middle Aged': {'yes': 4, 'no': 10},
 'Elderly': {'yes': 8, 'no': 6}}

In [4]:
json_path = _IMGS / "age_conclusions.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(age_conclusions, f, indent=4, sort_keys=True)

png_path = _IMGS / "age_conclusions.png"
svg_path = _IMGS / "age_conclusions.svg"

render_stacked_horizontal(json_path, [png_path, svg_path])

print(f"Wrote {json_path}")
print(f"Wrote {png_path}")
print(f"Wrote {svg_path}")

Wrote /Users/josh/Desktop/harvard/kempner/viz/conclusions/age_conclusions/v1/age_conclusions.json
Wrote /Users/josh/Desktop/harvard/kempner/viz/conclusions/age_conclusions/v3/age_conclusions.png
Wrote /Users/josh/Desktop/harvard/kempner/viz/conclusions/age_conclusions/v3/age_conclusions.svg
